# ✂️ [Colab 실습] 프루닝 첫걸음 — 순수 파이썬으로 밑바닥부터

**온디바이스 AI 프로그래밍 · 프루닝&지식증류 미니랩 들어가기 전 준비 실습**

| 항목 | 내용 |
| --- | --- |
| 대상 | 프루닝(가지치기)을 처음 접하는 분 (프레임워크 지식 불필요) |
| 도구 | **순수 파이썬 + numpy + matplotlib만** — PyTorch 없음! |
| 환경 | Google Colab CPU 런타임 |
| 진행 | 위에서부터 셀을 하나씩 실행 (`Shift + Enter`) |

## 이 실습의 목표

"가중치의 90%를 지워도 모델이 멀쩡하다"는 프루닝의 약속에는 함정이 있습니다.
**지워도 작아지지 않고, 빨라지지도 않는** 경우가 대부분이거든요.
이 실습에서는 그 함정을 직접 밟아본 뒤, 진짜 효과를 내는 **구조적 프루닝(연쇄 수술)**까지
순수 파이썬으로 전 과정을 만들어 봅니다.

## 로드맵

| Part | 주제 | 핵심 발견 |
| --- | --- | --- |
| 1 | 중요하지 않은 가중치가 정말 있는가 | 기여도 = \|w·x\| 실험 |
| 2 | 첫 가위질 — magnitude 프루닝 | 민감도 곡선 |
| 3 | 배신 — 0이 많아도 그대로다 | 크기·속도 실측 + 희소 저장의 손익 |
| 4 | 구조적 프루닝 — 뉴런째 연쇄 수술 | shape가 진짜 줄어든다 |
| 5 | 🏁 종합: 분류기 다이어트 | 하락 → 파인튜닝 회복 |
| 6 | 정리 — PyTorch 미니랩·NPU와의 연결 | 다음 실습 지도 |

> 💡 각 Step의 **✅ 확인**을 점검하고 **✏️ 직접 해보기**로 실험하세요.


---
# Part 0. 환경 준비

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
rng = np.random.default_rng(42)

try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"],
                   capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
    print("한글 폰트 설정 완료")
except Exception as e:
    print("한글 폰트 생략:", e)
plt.rc("axes", unicode_minus=False)
print("준비 완료 ✂️")

---
# Part 1. 중요하지 않은 가중치가 정말 있는가

프루닝의 대전제: **"학습된 신경망의 가중치 중 상당수는 없어도 그만"**.
믿기 전에 확인부터 합시다.

### Step 1-1. 뉴런 하나의 가중치 크기 분포 보기

학습된 가중치는 대부분 **0 근처에 몰려** 있습니다. 정말 그런지, 그리고
각 가중치가 출력에 실제로 얼마나 **기여**하는지 잽니다.

기여도의 간단한 정의: 입력이 여러 개 들어올 때 $|w_i \times x_i|$ 의 평균 — "이 가중치가 출력에 보태는 양"


In [ ]:
# 학습된 뉴런이라 상상할 가중치 16개 (0 근처에 몰린 분포)
w = rng.normal(0, 0.3, 16)
inputs = rng.normal(0, 1.0, (200, 16))        # 입력 200개

contrib = np.abs(inputs * w).mean(axis=0)     # 가중치별 평균 기여도

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].bar(range(16), np.abs(w)); axes[0].set_title("|w| — 가중치 크기")
axes[0].set_xlabel("가중치 번호")
axes[1].bar(range(16), contrib, color="#e8a33d"); axes[1].set_title("평균 기여도 |w·x|")
axes[1].set_xlabel("가중치 번호")
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

order = np.argsort(np.abs(w))
print(f"가장 작은 가중치 3개: 번호 {list(order[:3])} → 기여도 {np.round(contrib[order[:3]],4)}")
print(f"가장 큰   가중치 3개: 번호 {list(order[-3:])} → 기여도 {np.round(contrib[order[-3:]],4)}")
print()
print("✅ |w|가 작으면 기여도도 작다 — '크기(magnitude)'가 중요도의 좋은 대리 지표라는 근거!")

### Step 1-2. 제거 실험 — 작은 것 vs 큰 것을 지웠을 때

가중치 **하나**를 0으로 만들고 뉴런 출력이 얼마나 변하는지 비교합니다.

In [ ]:
def neuron(w_vec, x):
    return float(np.dot(w_vec, x))

y_orig = np.array([neuron(w, x) for x in inputs])

def erase_and_measure(idx):
    w_cut = w.copy(); w_cut[idx] = 0.0
    y_cut = np.array([neuron(w_cut, x) for x in inputs])
    return np.abs(y_cut - y_orig).mean()

small_idx = order[0]      # 가장 작은 |w|
big_idx   = order[-1]     # 가장 큰 |w|

err_small = erase_and_measure(small_idx)
err_big   = erase_and_measure(big_idx)
base = np.abs(y_orig).mean()

print(f"가장 작은 가중치(#{small_idx}, w={w[small_idx]:+.3f}) 제거 → 출력 변화 {err_small:.4f} (출력 평균의 {err_small/base*100:.1f}%)")
print(f"가장 큰   가중치(#{big_idx}, w={w[big_idx]:+.3f}) 제거 → 출력 변화 {err_big:.4f} (출력 평균의 {err_big/base*100:.1f}%)")
print(f"\n영향력 차이: {err_big/err_small:.0f}배")
print("✅ 같은 '한 개 제거'라도 어느 것을 지우느냐가 하늘과 땅 차이 — 그래서 '무엇을 자를지'가 프루닝의 절반입니다.")

> **✅ Part 1 확인**
> - [ ] 학습된 가중치가 0 근처에 몰려 있음을 봤다
> - [ ] |w|가 작으면 출력 기여도도 작음을 실험으로 확인했다
> - [ ] "무엇을 자를지"의 기준으로 magnitude(크기)가 합리적임을 이해했다

---
# Part 2. 첫 가위질 — magnitude 프루닝 함수 만들기

### Step 2-1. 순수 파이썬으로 prune 함수

규칙은 하나: **절대값이 작은 순서로 하위 p%를 0으로**.

In [ ]:
def prune(values, ratio):
    """순수 파이썬 magnitude 프루닝.
    values: 리스트, ratio: 0.0~1.0 (지울 비율) → 0이 섞인 새 리스트"""
    n_cut = int(len(values) * ratio)
    # 절대값 기준 오름차순 정렬해 '자를 경계값' 찾기
    threshold = sorted(abs(v) for v in values)[n_cut - 1] if n_cut > 0 else -1
    return [0.0 if abs(v) <= threshold else v for v in values]

demo = [0.5, -0.02, 0.31, 0.007, -0.9, 0.11, -0.04, 0.62]
for r in [0.25, 0.5, 0.75]:
    pruned = prune(demo, r)
    print(f"{int(r*100):>3}% 프루닝: {[round(v,3) for v in pruned]}  (0 개수: {pruned.count(0.0)})")
print()
print("✅ 작은 값부터 차례로 0이 된다 — 이것이 magnitude 프루닝의 전부!")

### Step 2-2. 레이어 전체에 적용 — 민감도 곡선 그리기

16→8 레이어(가중치 128개)를 10%~95% 프루닝하며 **출력 오차가 언제 무너지는지** 관찰합니다.

In [ ]:
W_layer = rng.normal(0, 0.3, (16, 8))
X_batch = rng.normal(0, 1.0, (200, 16))
Y_base = X_batch @ W_layer

ratios = [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]
errors = []
for r in ratios:
    Wp = np.array(prune(W_layer.flatten().tolist(), r)).reshape(16, 8)
    err = np.abs(X_batch @ Wp - Y_base).mean() / np.abs(Y_base).mean() * 100
    errors.append(err)
    print(f"{int(r*100):>3}% 프루닝 → 출력 상대 오차 {err:5.1f}%  (0 비율 {np.mean(Wp==0)*100:.0f}%)")

plt.figure(figsize=(7, 3.6))
plt.plot([r*100 for r in ratios], errors, "o-", lw=2)
plt.xlabel("프루닝 비율 (%)"); plt.ylabel("출력 상대 오차 (%)")
plt.title("민감도 곡선 — 절반을 지워도 오차는 완만, 어느 순간 급증")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print("\n💡 곡선의 '무릎'이 이 레이어의 한계 — 레이어마다 다르며, 이것이 '레이어별 민감도 분석'의 정체입니다.")

> ✏️ **직접 해보기:** `W_layer`를 `rng.normal(0, 0.3, ...)` 대신 `rng.uniform(0.2, 0.5, ...)`
> (0 근처가 **비어있는** 분포)로 바꿔 보세요. 민감도 곡선이 어떻게 달라지나요? 왜일까요?

> **✅ Part 2 확인**
> - [ ] magnitude 프루닝을 순수 파이썬 5줄로 구현했다
> - [ ] 민감도 곡선의 '완만 구간'과 '급증 구간'을 봤다
> - [ ] 프루닝 가능한 비율은 가중치 분포에 달려 있음을 이해했다

---
# Part 3. 배신 — 0이 많아도 작아지지도, 빨라지지도 않는다

90%를 0으로 만들었으니 크기 1/10, 속도 10배… 일까요? **직접 재봅시다.**

### Step 3-1. 크기와 속도 실측 — 충격의 결과

In [ ]:
W_dense  = rng.normal(0, 0.3, (512, 512)).astype(np.float32)
W_pruned = np.array(prune(W_dense.flatten().tolist(), 0.9), dtype=np.float32).reshape(512, 512)
X_big = rng.normal(0, 1, (64, 512)).astype(np.float32)

print(f"0의 비율   : 원본 {np.mean(W_dense==0)*100:.0f}%  vs 프루닝 {np.mean(W_pruned==0)*100:.0f}%")
print(f"메모리 크기: 원본 {W_dense.nbytes:,}B vs 프루닝 {W_pruned.nbytes:,}B  ← 똑같다?!")

def timeit(f, n=30):
    t0 = time.perf_counter()
    for _ in range(n): f()
    return (time.perf_counter() - t0) / n * 1000

t_dense  = timeit(lambda: X_big @ W_dense)
t_pruned = timeit(lambda: X_big @ W_pruned)
print(f"행렬곱 시간: 원본 {t_dense:.3f}ms vs 프루닝 {t_pruned:.3f}ms  ← 거의 똑같다?!")
print()
print("💥 배신의 이유:")
print("  ① 크기 — 0도 float32로 4바이트씩 '자리를 차지'하며 저장된다 (dense 형식)")
print("  ② 속도 — 하드웨어는 0인지 확인하지 않고 그냥 곱한다 (0×x도 곱셈 1회)")

### Step 3-2. 크기 문제의 해법 — 희소(sparse) 저장 직접 만들기

0을 저장하지 않으려면 **"몇 번째 자리에 어떤 값"** 목록으로 바꿔야 합니다.
공짜가 아닙니다 — 값(4B)마다 **위치 인덱스(4B)**가 따라붙습니다.

In [ ]:
def to_sparse(flat_values):
    """희소 형식: (인덱스, 값) 쌍의 리스트"""
    return [(i, v) for i, v in enumerate(flat_values) if v != 0.0]

n_total = W_pruned.size
n_nonzero = int(np.count_nonzero(W_pruned))

dense_bytes  = n_total * 4                       # 값 4B × 전부
sparse_bytes = n_nonzero * (4 + 4)               # (값 4B + 인덱스 4B) × 0 아닌 것만

print(f"전체 {n_total:,}개 중 0이 아닌 값 {n_nonzero:,}개 (90% 프루닝)")
print(f"dense  저장: {dense_bytes:,}B")
print(f"sparse 저장: {sparse_bytes:,}B  → {dense_bytes/sparse_bytes:.1f}배 절감 ✅")
print()
# 손익분기: sparse가 이득이려면?  nz×8 < n×4  →  nz < n/2
print("손익분기 계산: 0이 아닌 값 × 8B < 전체 × 4B  →  0이 아닌 비율 < 50%")
for zr in [0.3, 0.5, 0.7, 0.9]:
    nz = int(n_total * (1 - zr))
    ratio = dense_bytes / (nz * 8)
    mark = "이득 ✅" if ratio > 1 else "오히려 손해 ❌"
    print(f"  0 비율 {int(zr*100)}% → sparse가 dense의 {1/ratio*100:5.0f}% 크기 ({mark})")
print()
print("💡 '90%의 배신' 절반 해결 — 단, 인덱스 오버헤드 때문에 50%는 지워야 본전입니다.")

### Step 3-3. 속도 문제 — 0 건너뛰는 곱을 직접 구현해 3파전

이제 0을 **건너뛰는** 곱을 만들어 세 선수를 겨루게 합니다:
① 순수 파이썬 dense 루프(0도 곱함) ② 순수 파이썬 sparse 루프(0 스킵) ③ numpy dense(0도 곱하지만 벡터화)

In [ ]:
w_row = W_pruned[:, 0].tolist()               # 뉴런 하나의 가중치 512개 (90%가 0)
x_vec = X_big[0].tolist()
sparse_row = to_sparse(w_row)                 # (인덱스, 값) — 0 아닌 ~51개

def dot_dense_py():
    acc = 0.0
    for i in range(len(w_row)): acc += w_row[i] * x_vec[i]
    return acc

def dot_sparse_py():
    acc = 0.0
    for i, v in sparse_row: acc += v * x_vec[i]
    return acc

w_np = np.array(w_row, dtype=np.float32); x_np = np.array(x_vec, dtype=np.float32)
def dot_numpy(): return float(w_np @ x_np)

assert abs(dot_dense_py() - dot_sparse_py()) < 1e-6 and abs(dot_dense_py() - dot_numpy()) < 1e-3

t1 = timeit(dot_dense_py, 200); t2 = timeit(dot_sparse_py, 200); t3 = timeit(dot_numpy, 200)
print(f"① 파이썬 dense 루프 (512회 곱)  : {t1*1000:7.1f} µs")
print(f"② 파이썬 sparse 루프 ({len(sparse_row)}회 곱) : {t2*1000:7.1f} µs  ← 스킵 덕에 {t1/t2:.0f}배 빨라짐!")
print(f"③ numpy dense (512회 곱+벡터화): {t3*1000:7.1f} µs  ← 그래도 압승 ({t2/t3:.0f}배)")
print()
print("💡 결론: 0 스킵은 '같은 방식끼리'는 이득이지만, 불규칙한 인덱스 접근이")
print("   벡터화·캐시를 깨뜨려 고속 하드웨어에선 오히려 밀립니다.")
print("   → 비구조적 희소성으로 실속 가속을 얻으려면 BlackSwan의 Pruning Engine처럼")
print("     '0 스킵을 하드웨어가 직접' 지원해야 합니다 (교안 Day 2).")

> **✅ Part 3 확인**
> - [ ] 90% 프루닝 후에도 dense 크기·속도가 그대로임을 실측했다
> - [ ] sparse 저장(값+인덱스)을 직접 만들었고 손익분기 50%를 계산했다
> - [ ] 0 스킵 곱을 구현했지만 numpy dense에 밀리는 이유(불규칙 접근)를 설명할 수 있다
>
> **다음 질문:** 그럼 shape 자체를 줄여서 **모두에게 진짜로 작고 빠른** 프루닝은 없을까?
> → Part 4의 구조적 프루닝, 일명 **연쇄 수술**입니다.

---
# Part 4. 구조적 프루닝 — 뉴런째 잘라내는 연쇄 수술

가중치 낱개가 아니라 **뉴런(채널) 단위**로 잘라내면 행렬의 **shape 자체**가 줄어듭니다.
shape가 줄면 어떤 하드웨어에서든 진짜로 작아지고 빨라집니다.

### Step 4-1. 미니 2층 네트워크와 '뉴런 중요도'

```text
입력(4) ──W1(4×6)──▶ 은닉(6) ──W2(6×2)──▶ 출력(2)
```

은닉 뉴런 j를 지우려면 두 군데를 **함께** 잘라야 합니다 (그래서 '연쇄'):
- W1의 **j번째 열** (그 뉴런으로 들어가는 선들)
- W2의 **j번째 행** (그 뉴런에서 나가는 선들)

뉴런 중요도는 그 두 묶음의 크기로 잽니다: $imp_j = \sqrt{\|W1[:,j]\|^2 + \|W2[j,:]\|^2}$

In [ ]:
W1 = rng.normal(0, 0.5, (4, 6))
W2 = rng.normal(0, 0.5, (6, 2))
X_in = rng.normal(0, 1.0, (100, 4))

def forward2(X, W1_, W2_):
    h = np.maximum(0, X @ W1_)            # ReLU
    return h @ W2_

Y_base = forward2(X_in, W1, W2)

imp = np.sqrt((W1**2).sum(axis=0) + (W2**2).sum(axis=1))
print("은닉 뉴런 중요도:", np.round(imp, 3))
weak2 = np.argsort(imp)[:2]
print(f"→ 가장 약한 뉴런 2개: {sorted(weak2.tolist())}번")

plt.figure(figsize=(6, 2.8))
colors = ["#c44e52" if j in weak2 else "#4c72b0" for j in range(6)]
plt.bar(range(6), imp, color=colors)
plt.xlabel("은닉 뉴런 번호"); plt.ylabel("중요도 (L2)")
plt.title("뉴런 중요도 — 빨간 2개가 수술 대상")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Step 4-2. 연쇄 수술 집도 — shape가 진짜 줄어든다

In [ ]:
keep = np.sort(np.argsort(imp)[2:])          # 강한 4개만 유지

W1_cut = W1[:, keep]                          # ① 열 삭제 (들어가는 선)
W2_cut = W2[keep, :]                          # ② 행 삭제 (나가는 선) ← 연쇄!

print(f"W1: {W1.shape} → {W1_cut.shape}")
print(f"W2: {W2.shape} → {W2_cut.shape}")
print(f"파라미터: {W1.size + W2.size}개 → {W1_cut.size + W2_cut.size}개")
print(f"크기(FP32): {(W1.size+W2.size)*4}B → {(W1_cut.size+W2_cut.size)*4}B")

Y_cut = forward2(X_in, W1_cut, W2_cut)
err = np.abs(Y_cut - Y_base).mean() / np.abs(Y_base).mean() * 100
print(f"\n출력 상대 오차: {err:.1f}%  (약한 뉴런만 잘랐기에 이 정도로 방어)")
print()
print("✅ 0을 '표시'한 게 아니라 행렬이 '실제로' 작아졌다 — 어떤 하드웨어든 이득을 봅니다.")
print("⚠️ 주의: W1의 열과 W2의 행을 '같은 번호로' 함께 잘라야 함 — 하나만 자르면 shape이 안 맞아 터집니다!")

### Step 4-3. 속도까지 실측 — 이번엔 진짜다

큰 행렬(은닉 512 → 절반 256)로 확대해 시간을 재봅니다. Part 3의 배신과 대비되는 순간입니다.

In [ ]:
W1b = rng.normal(0, 0.3, (256, 512)).astype(np.float32)
W2b = rng.normal(0, 0.3, (512, 64)).astype(np.float32)
Xb  = rng.normal(0, 1, (64, 256)).astype(np.float32)

impb = np.sqrt((W1b**2).sum(0) + (W2b**2).sum(1))
keepb = np.sort(np.argsort(impb)[256:])       # 512 → 256 (50% 구조적 프루닝)
W1s, W2s = W1b[:, keepb], W2b[keepb, :]

t_full = timeit(lambda: np.maximum(0, Xb @ W1b) @ W2b)
t_slim = timeit(lambda: np.maximum(0, Xb @ W1s) @ W2s)

print(f"은닉 512 (원본)      : {t_full:.3f} ms | 파라미터 {(W1b.size+W2b.size):,}")
print(f"은닉 256 (구조적 50%): {t_slim:.3f} ms | 파라미터 {(W1s.size+W2s.size):,}")
print(f"\n속도 {t_full/t_slim:.1f}배 · 크기 {(W1b.size+W2b.size)/(W1s.size+W2s.size):.1f}배 — Part 3의 '변화 없음'과 정반대!")
print()
print("| 비교 | 비구조적 90% | 구조적 50% |")
print("|---|---|---|")
print("| 크기 | 그대로 (sparse 형식 써야 절감) | 즉시 절반 |")
print("| 속도 | 그대로 (전용 HW 필요) | 즉시 ~2배 |")
print("| 자유도 | 아무 가중치나 | 뉴런/채널 단위만 |")

> **✅ Part 4 확인**
> - [ ] 뉴런 중요도(들어오는 열 + 나가는 행의 L2)를 계산했다
> - [ ] W1 열 삭제 + W2 행 삭제를 '같은 번호로 함께' 하는 연쇄 수술을 집도했다
> - [ ] 구조적 프루닝만이 크기·속도를 즉시 줄임을 실측으로 확인했다

---
# Part 5. 🏁 종합 — 미니 분류기 다이어트 (하락 → 회복)

양자화 첫걸음 실습의 4×4 분류기를 3클래스(**0 / 1 / 7**)로 키우고,
**구조적 프루닝 → 정확도 하락 → 파인튜닝 회복**의 전체 사이클을 체험합니다.

### Step 5-1. 데이터와 모델 준비 (이 셀은 실행만 하면 됩니다)

In [ ]:
B0 = np.array([[1,1,1,1],[1,0,0,1],[1,0,0,1],[1,1,1,1]], float)   # 0
B1 = np.array([[0,0,1,0],[0,1,1,0],[0,0,1,0],[0,1,1,1]], float)   # 1
B7 = np.array([[1,1,1,1],[0,0,0,1],[0,0,1,0],[0,1,0,0]], float)   # 7
BASES = [B0, B1, B7]

rng = np.random.default_rng(42)               # 재현성 위해 시드 리셋
def make_data(n, noise=0.55):
    Xs, ys = [], []
    for _ in range(n):
        y = rng.integers(0, 3)
        Xs.append((BASES[y] + rng.normal(0, noise, (4,4))).reshape(-1)); ys.append(y)
    return np.array(Xs), np.array(ys)

Xtr, ytr = make_data(600); Xte, yte = make_data(300)

H = 10                                        # 은닉 뉴런 10개로 시작
W1 = rng.normal(0, 0.3, (16, H)); b1 = np.zeros(H)
W2 = rng.normal(0, 0.3, (H, 3));  b2 = np.zeros(3)

def net(X, W1_,b1_,W2_,b2_):
    h = np.maximum(0, X @ W1_ + b1_)
    return h @ W2_ + b2_

def train(W1_,b1_,W2_,b2_, epochs, lr=0.3):
    for _ in range(epochs):
        h = np.maximum(0, Xtr @ W1_ + b1_)
        logit = h @ W2_ + b2_
        p = np.exp(logit - logit.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
        d = (p - np.eye(3)[ytr]) / len(Xtr)
        gW2 = h.T @ d; gb2 = d.sum(0)
        dh = d @ W2_.T * (h > 0)
        gW1 = Xtr.T @ dh; gb1 = dh.sum(0)
        W2_ -= lr*gW2; b2_ -= lr*gb2; W1_ -= lr*gW1; b1_ -= lr*gb1
    return W1_, b1_, W2_, b2_

W1, b1, W2, b2 = train(W1, b1, W2, b2, 400)
def acc(W1_,b1_,W2_,b2_):
    return (net(Xte,W1_,b1_,W2_,b2_).argmax(1) == yte).mean() * 100

acc_fp = acc(W1, b1, W2, b2)
n_params = W1.size + b1.size + W2.size + b2.size
print(f"학습 완료 — 은닉 {H}개 분류기 | 파라미터 {n_params}개 | 테스트 정확도 {acc_fp:.1f}%")

fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, (img, y) in zip(axes, zip(Xte[:6], yte[:6])):
    ax.imshow(img.reshape(4,4), cmap="gray"); ax.set_title(f"정답 {y if y<2 else 7}", fontsize=9); ax.axis("off")
plt.suptitle("테스트 이미지 예시 (노이즈가 꽤 강함)"); plt.tight_layout(); plt.show()

### Step 5-2. 민감도 곡선 — 몇 개까지 잘라도 되는가

은닉 뉴런을 중요도 순으로 0~8개 잘라가며 정확도를 관찰합니다.

In [ ]:
imp = np.sqrt((W1**2).sum(0) + (W2**2).sum(1))
order = np.argsort(imp)                        # 약한 순

cuts = list(range(0, 9))
accs = []
for c in cuts:
    keep = np.sort(order[c:])
    accs.append(acc(W1[:, keep], b1[keep], W2[keep, :], b2))
    print(f"뉴런 {c}개 컷 (유지 {H-c:>2}개) → 정확도 {accs[-1]:5.1f}%")

plt.figure(figsize=(7, 3.4))
plt.plot(cuts, accs, "o-", lw=2)
plt.axhline(acc_fp, ls="--", c="gray", label=f"원본 {acc_fp:.1f}%")
plt.xlabel("잘라낸 은닉 뉴런 수"); plt.ylabel("정확도 (%)")
plt.title("구조적 프루닝 민감도 곡선")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("💡 Part 2의 가중치 민감도 곡선과 같은 모양 — 단위만 '뉴런'으로 바뀌었을 뿐입니다.")

### Step 5-3. 공격적 수술 (10→3) + 무작위 대조군 — '기준'의 가치 증명

뉴런 7개를 잘라 3개만 남깁니다. 같은 개수를 **무작위로** 자른 경우와 비교합니다.

In [ ]:
CUT = 7
keep_imp = np.sort(order[CUT:])                       # 중요도 기준 생존자 3명
W1p, b1p, W2p = W1[:, keep_imp], b1[keep_imp], W2[keep_imp, :]
acc_pruned = acc(W1p, b1p, W2p, b2)

rng_pick = np.random.default_rng(1)
keep_rand = np.sort(rng_pick.choice(H, H-CUT, replace=False))
acc_random = acc(W1[:, keep_rand], b1[keep_rand], W2[keep_rand, :], b2)

n_after = W1p.size + b1p.size + W2p.size + b2.size
print(f"원본 (은닉 10)        : 정확도 {acc_fp:5.1f}% | 파라미터 {n_params}")
print(f"중요도 기준 컷 (은닉 3): 정확도 {acc_pruned:5.1f}% | 파라미터 {n_after} ({n_params/n_after:.1f}배 축소)")
print(f"무작위 컷     (은닉 3): 정확도 {acc_random:5.1f}% ← 참사!")
print()
print(f"✅ 같은 '7개 자르기'인데 {acc_pruned-acc_random:.0f}%p 차이 — Part 1에서 세운 '중요도 기준'의 가치가 증명됐습니다.")
print(f"⚠️ 하지만 중요도 기준도 {acc_fp-acc_pruned:.1f}%p 하락 — 이제 회복시킬 차례입니다.")

### Step 5-4. 파인튜닝 — 남은 뉴런들이 빈자리를 메운다

수술 후 짧은 재학습(60 에폭)으로 남은 3개 뉴런이 역할을 재조정하게 합니다.

In [ ]:
W1f, b1f, W2f, b2f = train(W1p.copy(), b1p.copy(), W2p.copy(), b2.copy(), 60)
acc_ft = acc(W1f, b1f, W2f, b2f)

print("════ 다이어트 최종 리포트 ════")
print(f"{'단계':<24}{'정확도':>8}{'파라미터':>10}{'크기(FP32)':>12}")
print(f"{'① 원본 (은닉 10)':<23}{acc_fp:>7.1f}%{n_params:>10}{n_params*4:>10}B")
print(f"{'② 구조적 프루닝 (은닉 3)':<21}{acc_pruned:>7.1f}%{n_after:>10}{n_after*4:>10}B")
print(f"{'③ + 파인튜닝 60에폭':<22}{acc_ft:>7.1f}%{n_after:>10}{n_after*4:>10}B")
print()
print(f"결론: 파라미터 {n_params/n_after:.1f}배 줄이고 정확도 {acc_ft:.1f}% ({acc_ft-acc_fp:+.1f}%p) — ")
print("      '프루닝 → 파인튜닝' 2단 콤보가 표준 레시피인 이유입니다.")
print()
print("📎 교안 Day 2 결정 규칙과 동일: 프루닝 후 손실이 크면 → 파인튜닝(재학습)으로 회복 시도")

### Step 5-5. 리포트 과제 — 직접 실험하고 표를 채우세요

| 실험 | 조건 | 프루닝 직후 | 파인튜닝 후 | 관찰 |
| --- | --- | --- | --- | --- |
| 기준 | CUT=7 (위 그대로) | | | |
| 실험1 | CUT=8 (은닉 2개만 유지) | | | |
| 실험2 | 파인튜닝 60 → **10 에폭**만 | | | |
| 실험3 | 중요도를 $\|W1[:,j]\|$만으로 (나가는 행 무시) | | | |

**분석 질문 (2~3문장씩):**
1. 실험1에서 파인튜닝으로도 회복이 안 된다면, 그 이유는? (힌트: 3클래스를 뉴런 2개로 구분할 수 있는가)
2. 실험3의 중요도 기준은 왜 위험한가요? (힌트: 들어오는 선은 크지만 나가는 선이 0에 가까운 뉴런)
3. 실전에서 "프루닝 비율"을 정할 때 민감도 곡선을 어떻게 활용하겠습니까?

In [ ]:
# ✏️ 실험 공간 — 예시 (실험1):
CUT2 = 8
k2 = np.sort(order[CUT2:])
m2 = (W1[:, k2].copy(), b1[k2].copy(), W2[k2, :].copy(), b2.copy())
a_direct = acc(*m2)
m2f = train(*[a.copy() for a in m2], 60)
print(f"실험1 (은닉 2개): 직후 {a_direct:.1f}% → 파인튜닝 후 {acc(*m2f):.1f}%")
# 실험2·3은 직접 작성해 보세요!

---
# Part 6. 정리 — 오늘 만든 부품 ↔ PyTorch 프루닝 미니랩

| 오늘 직접 만든 것 | PyTorch 미니랩 / 실전에서의 이름 |
| --- | --- |
| `prune()` (하위 \|w\| p% → 0) | `torch.nn.utils.prune.l1_unstructured` |
| 민감도 곡선 (비율 vs 오차) | 레이어별 민감도 분석 (프루닝 비율 결정 근거) |
| Part 3의 배신 실측 | "비구조적 90%의 배신" 실험 (크기·속도 불변) |
| `to_sparse()` 값+인덱스 저장 | sparse 텐서 포맷 (COO/CSR) — 인덱스 오버헤드 동일 |
| 연쇄 수술 (열+행 동시 삭제) | structured pruning + 다음 층 in-channel 동기화 |
| 뉴런 중요도 (열+행 L2) | channel importance (L1/L2 norm 기준) |
| 프루닝 → 파인튜닝 콤보 | 표준 프루닝 파이프라인 (train → prune → finetune) |

## NPU와의 연결 (교안 Day 2)

- **BlackSwan Pruning Engine**: Part 3에서 본 "0 스킵"을 **하드웨어가 직접** 수행 —
  비구조적 희소성에서도 유효 연산이 줄어드는 이유. XWN과 결합해 2 TOPS를 만드는 핵심.
- **구조적 프루닝**은 NPU 유무와 무관하게 이득 — 컴파일 전에 채널을 줄여 두면
  타일 수 자체가 줄어듭니다 (「타일링」 애니메이션의 타일 개수 공식 참고).

## ✏️ 심화 도전 과제 (선택)

1. **반복 프루닝**: 한 번에 7개 대신 "1개 컷 → 10에폭 파인튜닝"을 7회 반복하면 최종 정확도가 더 좋은지 실험
2. **전역 vs 층별**: W1·W2를 합쳐 하위 50%를 지우는 전역 방식과, 각 층에서 50%씩 지우는 층별 방식의 오차 비교
3. **프루닝 + 양자화 콤보**: Part 5의 3뉴런 모델을 「양자화 첫걸음」의 `quantize()`로 INT8화 —
   압축률이 몇 배까지 가는지 계산 (구조적 3.2배 × 양자화 4배 = ?)
4. **복권 가설 맛보기**: 파인튜닝 대신 '살아남은 구조'를 처음부터 재학습하면 어떻게 되는지 실험

---

수고하셨습니다! 🎉 이제 프루닝을 "지우면 빨라지는 마법"이 아니라
**"무엇을·어떤 단위로 지우고, 어떻게 회복시키느냐의 공학"**으로 이해하게 되었습니다.
다음 실습 「프루닝&지식증류 미니랩」에서 같은 이야기가 CNN 규모로 펼쳐집니다.
